# Praktikum UTS – Klasifikasi Menggunakan Naive Bayes

**NIM: 202431148**

Notebook ini mengolah empat dataset menggunakan algoritma **Naive Bayes (GaussianNB)**. Tahapan setiap dataset meliputi pemeriksaan data, EDA, preprocessing, pembagian data training/testing, pelatihan model, dan evaluasi.

**Petunjuk:** Simpan notebook ini dalam folder yang sama dengan keempat file CSV. Jalankan cell secara berurutan dari atas ke bawah.

## 1. Import Library

In [ ]:
# Watermark NIM: 202431148
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Library berhasil diimpor.")

## 2. Fungsi untuk EDA, Preprocessing, Training, dan Evaluasi

Fungsi berikut digunakan kembali pada empat dataset agar langkah pengolahan konsisten. Kolom numerik akan diisi nilai kosong menggunakan median dan distandardisasi. Kolom kategorikal akan diisi menggunakan modus lalu diubah menjadi one-hot encoding. Seluruh preprocessing dipasang di dalam pipeline sehingga parameter preprocessing dipelajari hanya dari data training untuk menghindari data leakage.

In [ ]:
# Watermark NIM: 202431148
def buat_onehot_encoder():
    # Mendukung beberapa versi scikit-learn
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def analisis_naive_bayes(df, nama_dataset, target):
    print("=" * 80)
    print(f"DATASET: {nama_dataset}")
    print("=" * 80)

    # Salin data agar data asli tidak berubah
    data = df.copy()

    # Rapikan nama kolom dan hapus baris duplikat
    data.columns = data.columns.astype(str).str.strip()
    jumlah_awal = len(data)
    data = data.drop_duplicates().reset_index(drop=True)
    print(f"Jumlah baris awal       : {jumlah_awal}")
    print(f"Duplikat yang dihapus   : {jumlah_awal - len(data)}")
    print(f"Ukuran data setelah cek : {data.shape}")

    if target not in data.columns:
        raise ValueError(f"Kolom target '{target}' tidak ditemukan. Kolom tersedia: {list(data.columns)}")

    # Hapus baris yang target/label-nya kosong karena tidak dapat digunakan untuk supervised learning
    data = data.dropna(subset=[target]).copy()

    print("\n--- Informasi Dataset ---")
    display(data.head())
    print("\nTipe data:")
    display(data.dtypes.astype(str).to_frame("Tipe Data"))
    print("\nStatistik deskriptif:")
    display(data.describe(include="all").transpose())
    print("\nJumlah nilai kosong per kolom:")
    display(data.isnull().sum().to_frame("Jumlah Missing Value"))

    # Distribusi kelas target
    print("\nDistribusi kelas target:")
    display(data[target].value_counts(dropna=False).to_frame("Jumlah Data"))
    plt.figure(figsize=(6, 4))
    sns.countplot(data=data, x=target, order=data[target].value_counts().index)
    plt.title(f"Distribusi Target - {nama_dataset}")
    plt.xlabel("Kelas Target")
    plt.ylabel("Jumlah")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

    # Histogram untuk kolom numerik
    kolom_numerik_eda = data.drop(columns=[target]).select_dtypes(include=np.number).columns.tolist()
    if kolom_numerik_eda:
        data[kolom_numerik_eda].hist(figsize=(14, 9), bins=20, edgecolor="black")
        plt.suptitle(f"Distribusi Fitur Numerik - {nama_dataset}")
        plt.tight_layout()
        plt.show()

    # Pisahkan fitur dan target
    X = data.drop(columns=[target])
    y = data[target]

    # Kolom numerik dan kategorikal ditentukan dari tipe data
    kolom_numerik = X.select_dtypes(include=np.number).columns.tolist()
    kolom_kategorikal = X.select_dtypes(exclude=np.number).columns.tolist()

    print("\nFitur numerik:", kolom_numerik)
    print("Fitur kategorikal:", kolom_kategorikal)

    # Preprocessing fitur numerik
    proses_numerik = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Preprocessing fitur kategorikal
    proses_kategorikal = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", buat_onehot_encoder())
    ])

    daftar_transformer = []
    if kolom_numerik:
        daftar_transformer.append(("numerik", proses_numerik, kolom_numerik))
    if kolom_kategorikal:
        daftar_transformer.append(("kategorikal", proses_kategorikal, kolom_kategorikal))

    preprocessing = ColumnTransformer(
        transformers=daftar_transformer,
        remainder="drop"
    )

    # Bagi data menjadi 80% training dan 20% testing
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )

    print(f"\nJumlah data training: {len(X_train)}")
    print(f"Jumlah data testing : {len(X_test)}")

    # Pipeline preprocessing + model Gaussian Naive Bayes
    model = Pipeline(steps=[
        ("preprocessing", preprocessing),
        ("naive_bayes", GaussianNB())
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Evaluasi model
    akurasi = accuracy_score(y_test, y_pred)
    presisi = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    print("\n--- Hasil Evaluasi Naive Bayes ---")
    print(f"Akurasi  : {akurasi:.4f} ({akurasi * 100:.2f}%)")
    print(f"Presisi  : {presisi:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    # Confusion matrix
    label_kelas = model.named_steps["naive_bayes"].classes_
    cm = confusion_matrix(y_test, y_pred, labels=label_kelas)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=label_kelas, yticklabels=label_kelas
    )
    plt.title(f"Confusion Matrix - {nama_dataset}")
    plt.xlabel("Prediksi")
    plt.ylabel("Aktual")
    plt.tight_layout()
    plt.show()

    hasil = {
        "Dataset": nama_dataset,
        "Jumlah Data": len(data),
        "Jumlah Fitur": X.shape[1],
        "Akurasi": akurasi,
        "Presisi": presisi,
        "Recall": recall,
        "F1-score": f1
    }
    return model, hasil

## 3. Dataset 1 – Heart Disease

Dataset ini digunakan untuk memprediksi apakah seseorang memiliki penyakit jantung. Kolom target yang digunakan adalah `HeartDisease`.

In [ ]:
# Watermark NIM: 202431148
heart_df = pd.read_csv("01_heart_disease.csv")
model_heart, hasil_heart = analisis_naive_bayes(
    heart_df,
    nama_dataset="Heart Disease",
    target="HeartDisease"
)

## 4. Dataset 2 – Diabetes

Dataset ini digunakan untuk memprediksi status diabetes. Kolom target yang digunakan adalah `diabetes`.

In [ ]:
# Watermark NIM: 202431148
diabetes_df = pd.read_csv("02_diabetes.csv")
model_diabetes, hasil_diabetes = analisis_naive_bayes(
    diabetes_df,
    nama_dataset="Diabetes",
    target="diabetes"
)

## 5. Dataset 3 – Counter-Strike 2

Dataset ini digunakan untuk memprediksi tim pemenang ronde permainan. Kolom target yang digunakan adalah `round_winner`.

In [ ]:
# Watermark NIM: 202431148
cs2_df = pd.read_csv("03_counter_strike_2.csv")
model_cs2, hasil_cs2 = analisis_naive_bayes(
    cs2_df,
    nama_dataset="Counter-Strike 2",
    target="round_winner"
)

## 6. Dataset 4 – Bank Customers

File Bank Customers menggunakan pemisah titik koma (`;`), sehingga harus dibaca dengan parameter `sep=";"`. Target yang digunakan adalah `y`, yaitu hasil/keputusan berlangganan deposito.

In [ ]:
# Watermark NIM: 202431148
bank_df = pd.read_csv("04_bank_customers.csv", sep=";")
model_bank, hasil_bank = analisis_naive_bayes(
    bank_df,
    nama_dataset="Bank Customers",
    target="y"
)

## 7. Perbandingan Hasil Keempat Dataset

Tabel berikut merangkum hasil evaluasi model Naive Bayes pada seluruh dataset. Nilai akurasi, presisi, recall, dan F1-score berasal dari data testing.

In [ ]:
# Watermark NIM: 202431148
tabel_hasil = pd.DataFrame([
    hasil_heart,
    hasil_diabetes,
    hasil_cs2,
    hasil_bank
])

display(
    tabel_hasil.style.format({
        "Akurasi": "{:.4f}",
        "Presisi": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}"
    })
)

plt.figure(figsize=(10, 5))
sns.barplot(data=tabel_hasil, x="Dataset", y="Akurasi")
plt.title("Perbandingan Akurasi Naive Bayes pada Empat Dataset")
plt.ylabel("Akurasi")
plt.xlabel("Dataset")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 8. Kesimpulan Otomatis Berdasarkan Hasil

Kesimpulan ini dibuat dari hasil evaluasi yang benar-benar diperoleh setelah seluruh cell dijalankan.

In [ ]:
# Watermark NIM: 202431148
terbaik = tabel_hasil.loc[tabel_hasil["Akurasi"].idxmax()]
terendah = tabel_hasil.loc[tabel_hasil["Akurasi"].idxmin()]

print("HASIL AKHIR PRAKTIKUM NAIVE BAYES")
print("-" * 50)
print(
    f"Akurasi tertinggi diperoleh dataset {terbaik['Dataset']} "
    f"dengan akurasi {terbaik['Akurasi'] * 100:.2f}%."
)
print(
    f"Akurasi terendah diperoleh dataset {terendah['Dataset']} "
    f"dengan akurasi {terendah['Akurasi'] * 100:.2f}%."
)
print(
    "\nPerbedaan hasil dapat dipengaruhi oleh jumlah data, keseimbangan kelas, "
    "karakteristik fitur, serta hubungan antara fitur dan label."
)
print(
    "Naive Bayes menggunakan prinsip probabilitas dan asumsi independensi "
    "antarfitur. Karena asumsi tersebut tidak selalu sepenuhnya terpenuhi, "
    "hasil model perlu dinilai menggunakan lebih dari satu metrik evaluasi."
)

---
**Akhir notebook – NIM 202431148**

Catatan: Jalankan seluruh cell dari awal sampai akhir agar tabel perbandingan dan kesimpulan dapat ditampilkan. Hasil evaluasi dapat berbeda jika dataset atau pengaturan pembagian data diubah.